# Gap-aware graph feature extraction from PLI and wPLI

This notebook converts the validated PLI/wPLI connectivity matrices into graph-theoretical features while preserving the authoritative epoch metadata from the Handcrafted CSV files.

**Primary analysis**

- 18 nodes = the 18 bipolar EEG derivations used during segmentation.
- Five bands: delta, theta, alpha, beta and gamma.
- Two connectivity measures: PLI and wPLI.
- Epoch-specific threshold: $T_e=0.5\max_{i\ne j}(W_{ij}^{(e)})$.
- Strong edges are retained with $W_{ij}\ge T_e$.
- Weighted path length is $1/W_{ij}$.
- One patient is processed per Kaggle run by default.
- Outputs are compressed Parquet files, not CSV.

The notebook never infers preictal/interictal labels from connectivity arrays. Training and continuous-testing metadata are copied from the already validated Handcrafted tables.


In [ ]:
# ==========================================================================
# 0. IMPORTS AND REPRODUCIBLE CONFIGURATION
# ==========================================================================
import gc
import json
import os
import re
import shutil
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
from numba import njit, prange
from IPython.display import display

CHANNELS_18 = [
    'FP1-F7', 'F7-T7', 'T7-P7', 'P7-O1',
    'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1',
    'FP2-F4', 'F4-C4', 'C4-P4', 'P4-O2',
    'FP2-F8', 'F8-T8', 'T8-P8', 'P8-O2',
    'FZ-CZ', 'CZ-PZ',
]
BANDS = ('delta', 'theta', 'alpha', 'beta', 'gamma')
MEASURES = ('PLI', 'wPLI')

# Fixed a priori for the primary experiment. Do not tune this on the test data.
THRESHOLD_RATIO = 0.5
CHUNK_SIZE = 256
EPSILON = 1e-12

# Kaggle input/output control.
INPUT_SEARCH_ROOT = Path('/kaggle/input')
INPUT_ROOT_OVERRIDE = None  # e.g. Path('/kaggle/input/datasets/ngoclannguyen00/handcrafted-connectivity')
OUTPUT_ROOT = Path('/kaggle/working/GraphFeatures_GAPAWARE_V3')

RUN_ALL_SUBJECTS = True      # Primary mode: process the complete eligible cohort in one Run All.
PATIENTS_PER_BATCH = 1       # Used only when RUN_ALL_SUBJECTS=False.
BATCH_NUMBER = 0             # Used only when RUN_ALL_SUBJECTS=False.
SUBJECTS_TO_RUN = None       # e.g. ['chb01']; overrides RUN_ALL_SUBJECTS.
RESET_OUTPUT = True          # Safe on Kaggle: /kaggle/working is fresh for each run.

META_CANDIDATES = [
    'Patient', 'Class', 'SeizureID', 'Block', 'Band', 'Epoch',
    'Phase', 'TargetPreictal', 'EligibleForFAR', 'Discontinuity',
    'EpochStart', 'NpyRelativePath', 'EpochInFile',
]

def channel_token(name):
    return re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_')

CHANNEL_TOKENS = [channel_token(ch) for ch in CHANNELS_18]


In [ ]:
# ==========================================================================
# 1. DISCOVER AND VALIDATE THE ATTACHED KAGGLE DATASET
# ==========================================================================
def discover_input_roots(search_root=INPUT_SEARCH_ROOT, override=INPUT_ROOT_OVERRIDE):
    candidates = []
    if override is not None:
        candidates = [Path(override)]
    else:
        # The Kaggle mount may include /datasets/<owner>/<slug> or only /<slug>.
        for conn in Path(search_root).rglob('CONNECTIVITY'):
            candidates.append(conn.parent)

    valid = []
    for root in candidates:
        handcrafted = root / 'Handcrafted'
        connectivity = root / 'CONNECTIVITY'
        required = [
            handcrafted,
            connectivity / 'Training' / 'PLI_matrix',
            connectivity / 'Training' / 'wPLI_matrix',
            connectivity / 'Testing' / 'PLI_matrix',
            connectivity / 'Testing' / 'wPLI_matrix',
        ]
        if all(p.exists() for p in required):
            valid.append(root.resolve())

    valid = sorted(set(valid))
    if not valid:
        raise FileNotFoundError(
            'Could not find a dataset containing Handcrafted/ and '
            'CONNECTIVITY/{Training,Testing}/{PLI,wPLI}_matrix under /kaggle/input.')
    if len(valid) > 1:
        raise RuntimeError(f'Multiple complete input roots found: {valid}. Set INPUT_ROOT_OVERRIDE.')
    root = valid[0]
    return root, root / 'Handcrafted', root / 'CONNECTIVITY'


def discover_subjects(handcrafted_root, connectivity_root):
    handcrafted = {p.name for p in handcrafted_root.iterdir()
                   if p.is_dir() and re.fullmatch(r'chb\d{2}', p.name.lower())}
    per_tree = []
    for split in ('Training', 'Testing'):
        for measure in MEASURES:
            root = connectivity_root / split / f'{measure}_matrix'
            per_tree.append({p.name for p in root.iterdir()
                             if p.is_dir() and re.fullmatch(r'chb\d{2}', p.name.lower())})
    common = handcrafted.intersection(*per_tree)
    if not common:
        raise RuntimeError('No subject is complete in Handcrafted, PLI and wPLI trees.')
    return sorted(common)


def normalise_seizure_id(series):
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(int)
    out = series.astype(str).str.extract(r'(\d+)(?!.*\d)', expand=False)
    if out.isna().any():
        raise ValueError('A SeizureID value does not contain an integer.')
    return out.astype(int)


def load_authoritative_metadata(csv_path, subject):
    df = pd.read_csv(csv_path)
    required = {'Band', 'SeizureID', 'Epoch'}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f'{csv_path}: missing metadata columns {sorted(missing)}')
    df = df.copy()
    df['Band'] = df['Band'].astype(str).str.strip().str.lower()
    df['SeizureID'] = normalise_seizure_id(df['SeizureID'])
    if 'Patient' not in df.columns:
        df.insert(0, 'Patient', subject)
    else:
        df['Patient'] = subject
    if not set(df['Band']).issubset(BANDS):
        raise ValueError(f'{csv_path}: unexpected bands {sorted(set(df.Band).difference(BANDS))}')
    keep = [c for c in META_CANDIDATES if c in df.columns]
    return df[keep].copy()


In [ ]:
# ==========================================================================
# 2. EXACT WEIGHTED SHORTEST-PATH AND BRANDES CENTRALITY (NUMBA)
# ==========================================================================
@njit(parallel=True)
def _path_metrics_batch(W):
    '''Metrics for thresholded nonnegative undirected weighted graphs.

    Edge length is 1/weight. Betweenness uses weighted Brandes and is
    normalised to [0, 1] for an undirected graph. Characteristic path length is
    computed only inside the largest connected component. If that component
    has one node, CPL is stored as 0 and Path_Length_Defined is 0.
    '''
    n_epochs, n, _ = W.shape
    betweenness = np.zeros((n_epochs, n), dtype=np.float64)
    efficiency = np.zeros(n_epochs, dtype=np.float64)
    cpl_lcc = np.zeros(n_epochs, dtype=np.float64)
    lcc_fraction = np.zeros(n_epochs, dtype=np.float64)
    n_components = np.zeros(n_epochs, dtype=np.float64)
    path_defined = np.zeros(n_epochs, dtype=np.float64)

    for e in prange(n_epochs):
        # Connected components and largest connected component.
        labels = np.full(n, -1, dtype=np.int64)
        sizes = np.zeros(n, dtype=np.int64)
        queue = np.empty(n, dtype=np.int64)
        comp = 0
        for start in range(n):
            if labels[start] != -1:
                continue
            head, tail = 0, 1
            queue[0] = start
            labels[start] = comp
            size = 0
            while head < tail:
                v = queue[head]
                head += 1
                size += 1
                for w in range(n):
                    if w != v and W[e, v, w] > 0.0 and labels[w] == -1:
                        labels[w] = comp
                        queue[tail] = w
                        tail += 1
            sizes[comp] = size
            comp += 1

        largest_label = 0
        largest_size = sizes[0]
        for c in range(1, comp):
            if sizes[c] > largest_size:
                largest_size = sizes[c]
                largest_label = c
        n_components[e] = comp
        lcc_fraction[e] = largest_size / n
        if largest_size >= 2:
            path_defined[e] = 1.0

        eff_sum = 0.0
        lcc_dist_sum = 0.0

        # Weighted Brandes: one Dijkstra search from every source.
        for source in range(n):
            dist = np.full(n, np.inf, dtype=np.float64)
            sigma = np.zeros(n, dtype=np.float64)
            delta = np.zeros(n, dtype=np.float64)
            visited = np.zeros(n, dtype=np.uint8)
            pred = np.zeros((n, n), dtype=np.uint8)
            order = np.empty(n, dtype=np.int64)
            order_count = 0
            dist[source] = 0.0
            sigma[source] = 1.0

            for _ in range(n):
                v = -1
                best = np.inf
                for node in range(n):
                    if visited[node] == 0 and dist[node] < best:
                        best = dist[node]
                        v = node
                if v == -1:
                    break
                visited[v] = 1
                order[order_count] = v
                order_count += 1

                for w in range(n):
                    weight = W[e, v, w]
                    if w == v or weight <= 0.0:
                        continue
                    alt = dist[v] + 1.0 / weight
                    tol = 1e-10 * max(1.0, abs(alt))
                    if alt < dist[w] - tol:
                        dist[w] = alt
                        sigma[w] = sigma[v]
                        for p in range(n):
                            pred[w, p] = 0
                        pred[w, v] = 1
                    elif abs(alt - dist[w]) <= tol:
                        sigma[w] += sigma[v]
                        pred[w, v] = 1

            for target in range(n):
                if target == source or not np.isfinite(dist[target]):
                    continue
                eff_sum += 1.0 / dist[target]
                if labels[source] == largest_label and labels[target] == largest_label:
                    lcc_dist_sum += dist[target]

            for oi in range(order_count - 1, -1, -1):
                w = order[oi]
                if sigma[w] > 0.0:
                    for v in range(n):
                        if pred[w, v] != 0:
                            delta[v] += (sigma[v] / sigma[w]) * (1.0 + delta[w])
                if w != source:
                    betweenness[e, w] += delta[w]

        efficiency[e] = eff_sum / (n * (n - 1))
        if largest_size >= 2:
            cpl_lcc[e] = lcc_dist_sum / (largest_size * (largest_size - 1))
        if n > 2:
            betweenness[e] /= ((n - 1) * (n - 2))

    return (betweenness, efficiency, cpl_lcc, lcc_fraction,
            n_components, path_defined)


In [ ]:
# ==========================================================================
# 3. GRAPH CONSTRUCTION AND FEATURE EXTRACTION
# ==========================================================================
def validate_connectivity_chunk(W, context='connectivity'):
    W = np.asarray(W)
    if W.ndim != 3 or W.shape[1:] != (18, 18):
        raise ValueError(f'{context}: expected (E,18,18), got {W.shape}')
    if not np.isfinite(W).all():
        raise ValueError(f'{context}: NaN or Inf in connectivity matrix')
    if W.min(initial=0.0) < -1e-6 or W.max(initial=0.0) > 1.0 + 1e-6:
        raise ValueError(f'{context}: connectivity values outside [0,1]')
    symmetry_error = float(np.max(np.abs(W - W.transpose(0, 2, 1)), initial=0.0))
    if symmetry_error > 1e-5:
        raise ValueError(f'{context}: maximum symmetry error {symmetry_error}')


def _weighted_clustering(Wt):
    '''Onnela/Rubinov-Sporns weighted clustering on thresholded graphs.'''
    max_w = Wt.max(axis=(1, 2), keepdims=True)
    normalised = np.divide(Wt, max_w, out=np.zeros_like(Wt), where=max_w > 0)
    root = np.cbrt(normalised)
    triangles_twice = np.einsum('eij,ejk,eki->ei', root, root, root, optimize=True)
    degree = (Wt > 0).sum(axis=2).astype(np.float64)
    denom = degree * (degree - 1.0)
    return np.divide(triangles_twice, denom,
                     out=np.zeros_like(triangles_twice), where=denom > 0)


def _eigenvector_centrality(W):
    if len(W) == 0:
        return np.empty((0, W.shape[-1]), dtype=np.float64)
    _, vectors = np.linalg.eigh(W)
    principal = np.abs(vectors[:, :, -1])
    norms = np.linalg.norm(principal, axis=1, keepdims=True)
    result = np.divide(principal, norms, out=np.zeros_like(principal), where=norms > 0)
    zero_graph = W.max(axis=(1, 2)) <= EPSILON
    result[zero_graph] = 0.0
    return result


def compute_graph_features(W, threshold_ratio=THRESHOLD_RATIO):
    W = np.asarray(W, dtype=np.float64).copy()
    validate_connectivity_chunk(W)
    W = np.clip((W + W.transpose(0, 2, 1)) / 2.0, 0.0, 1.0)
    diagonal = np.arange(W.shape[1])
    W[:, diagonal, diagonal] = 0.0

    offdiag_max = W.max(axis=(1, 2), keepdims=True)
    threshold = threshold_ratio * offdiag_max
    Wt = np.where((W >= threshold) & (W > 0.0), W, 0.0)
    Wt[:, diagonal, diagonal] = 0.0

    degree = (Wt > 0).sum(axis=2).astype(np.float64)
    strength = W.sum(axis=2)
    clustering = _weighted_clustering(Wt)
    eigenvector = _eigenvector_centrality(W)
    (betweenness, global_efficiency, cpl_lcc, lcc_fraction,
     n_components, path_defined) = _path_metrics_batch(Wt)

    n = W.shape[1]
    density = np.triu(Wt > 0, k=1).sum(axis=(1, 2)) / (n * (n - 1) / 2.0)

    columns = {}
    for metric, values in (
        ('Degree', degree),
        ('Strength', strength),
        ('Weighted_Clustering', clustering),
        ('Eigenvector_Centrality', eigenvector),
        ('Weighted_Betweenness', betweenness),
    ):
        for idx, token in enumerate(CHANNEL_TOKENS):
            columns[f'{token}_{metric}'] = values[:, idx].astype(np.float32)

    columns.update({
        'Density': density.astype(np.float32),
        'Mean_Weighted_Clustering': clustering.mean(axis=1).astype(np.float32),
        'Weighted_Global_Efficiency': global_efficiency.astype(np.float32),
        'Characteristic_Path_Length_LCC': cpl_lcc.astype(np.float32),
        'Largest_Component_Fraction': lcc_fraction.astype(np.float32),
        'Number_Connected_Components': n_components.astype(np.float32),
        'Path_Length_Defined': path_defined.astype(np.float32),
    })
    return pd.DataFrame(columns)


In [ ]:
# ==========================================================================
# 4. MAP EACH CONNECTIVITY FILE TO ITS AUTHORITATIVE METADATA ROWS
# ==========================================================================
def natural_key(path):
    return [(0, int(x)) if x.isdigit() else (1, x.lower())
            for x in re.findall(r'\d+|\D+', str(path))]


def seizure_from_filename(path):
    match = re.search(r'seizure_(\d+)', Path(path).stem)
    return int(match.group(1)) if match else None


def block_from_filename(path):
    match = re.search(r'_block_(\d+)', Path(path).stem)
    return int(match.group(1)) if match else None


def connectivity_files_for_table(csv_path, measure, connectivity_root, subject, band):
    stem = Path(csv_path).stem
    band = band.lower()
    if stem == 'preictal':
        root = connectivity_root / 'Training' / f'{measure}_matrix' / subject / 'preictal'
        return sorted(root.glob(f'*/{band}.npy'), key=natural_key)

    seizure_id = seizure_from_filename(csv_path)
    if seizure_id is None:
        raise ValueError(f'Cannot infer seizure number from {csv_path}')

    if stem.endswith('_interictal'):
        root = (connectivity_root / 'Training' / f'{measure}_matrix' / subject /
                'interictal' / f'seizure_{seizure_id:02d}')
        return sorted(root.glob(f'*/{band}.npy'), key=natural_key)

    block_id = block_from_filename(csv_path)
    if block_id is None:
        raise ValueError(f'Cannot infer block number from {csv_path}')
    root = (connectivity_root / 'Testing' / f'{measure}_matrix' / subject /
            f'seizure_{seizure_id:02d}' / f'block_{block_id:02d}')
    return sorted(root.glob(f'*/{band}.npy'), key=natural_key)


def ordered_metadata_for_band(meta, csv_path, band):
    rows = meta.loc[meta['Band'] == band].copy()
    expected_seizure = seizure_from_filename(csv_path)
    if expected_seizure is not None:
        rows = rows.loc[rows['SeizureID'] == expected_seizure]
    expected_block = block_from_filename(csv_path)
    if expected_block is not None and 'Block' in rows.columns:
        rows = rows.loc[rows['Block'].astype(int) == expected_block]
    # Epoch is authoritative within a seizure. Preictal.csv contains several
    # seizures whose Epoch counters each restart at zero, so SeizureID must be
    # the primary ordering key before matrices are matched sequentially.
    order = [c for c in ('SeizureID', 'Block', 'Epoch') if c in rows.columns]
    rows = rows.sort_values(order, kind='stable').reset_index(drop=True)
    return rows


def extract_one_band(files, metadata_rows, measure, band, chunk_size=CHUNK_SIZE):
    if not files:
        raise FileNotFoundError(f'No {measure}/{band} connectivity arrays found')
    lengths = []
    for path in files:
        array = np.load(path, mmap_mode='r', allow_pickle=False)
        if array.ndim != 3 or array.shape[1:] != (18, 18):
            raise ValueError(f'{path}: expected (E,18,18), got {array.shape}')
        lengths.append(int(array.shape[0]))
        del array
    if sum(lengths) != len(metadata_rows):
        raise AssertionError(
            f'{measure}/{band}: connectivity epochs={sum(lengths)} but metadata rows={len(metadata_rows)}; '
            f'files={files}')

    outputs = []
    cursor = 0
    for path, n_epochs in zip(files, lengths):
        array = np.load(path, mmap_mode='r', allow_pickle=False)
        file_meta = metadata_rows.iloc[cursor:cursor + n_epochs].reset_index(drop=True)
        for start in range(0, n_epochs, chunk_size):
            stop = min(start + chunk_size, n_epochs)
            graph = compute_graph_features(array[start:stop])
            meta_chunk = file_meta.iloc[start:stop].reset_index(drop=True).copy()
            meta_chunk['Measure'] = measure
            outputs.append(pd.concat([meta_chunk, graph], axis=1))
        cursor += n_epochs
        del array
        gc.collect()
    return pd.concat(outputs, ignore_index=True)


In [ ]:
# ==========================================================================
# 5. PATIENT PROCESSING, PARQUET OUTPUT AND QC
# ==========================================================================
def feature_columns(df):
    return [c for c in df.columns if c not in set(META_CANDIDATES + ['Measure'])]


def validate_graph_table(df, source_csv, measure):
    feats = feature_columns(df)
    values = df[feats].to_numpy(dtype=np.float64, copy=False)
    if not np.isfinite(values).all():
        raise ValueError(f'{source_csv}/{measure}: graph table contains NaN or Inf')

    key_candidates = ['Patient', 'Class', 'SeizureID', 'Block', 'Band', 'Epoch', 'Measure']
    keys = [c for c in key_candidates if c in df.columns]
    duplicates = int(df.duplicated(keys).sum())
    if duplicates:
        raise ValueError(f'{source_csv}/{measure}: {duplicates} duplicate graph keys {keys}')

    checks = {
        'Degree': (0.0, 17.0),
        'Strength': (0.0, 17.0),
        'Weighted_Clustering': (0.0, 1.0),
        'Eigenvector_Centrality': (0.0, 1.0),
        'Weighted_Betweenness': (0.0, 1.0),
        'Density': (0.0, 1.0),
        'Mean_Weighted_Clustering': (0.0, 1.0),
        'Weighted_Global_Efficiency': (0.0, 1.0),
        'Largest_Component_Fraction': (1.0 / 18.0, 1.0),
        'Path_Length_Defined': (0.0, 1.0),
    }
    tolerance = 1e-5
    for token, (lower, upper) in checks.items():
        cols = [c for c in feats if c == token or c.endswith('_' + token)]
        if not cols:
            raise AssertionError(f'Missing expected graph feature group: {token}')
        block = df[cols].to_numpy(dtype=np.float64, copy=False)
        if block.min(initial=lower) < lower - tolerance or block.max(initial=upper) > upper + tolerance:
            raise ValueError(f'{source_csv}/{measure}: {token} outside [{lower},{upper}]')
    return duplicates


def process_table(csv_path, subject, measure, handcrafted_root, connectivity_root, output_root):
    metadata = load_authoritative_metadata(csv_path, subject)
    band_outputs = []
    band_counts = {}
    for band in BANDS:
        rows = ordered_metadata_for_band(metadata, csv_path, band)
        if rows.empty:
            raise ValueError(f'{csv_path}: no authoritative rows for band={band}')
        files = connectivity_files_for_table(csv_path, measure, connectivity_root, subject, band)
        out = extract_one_band(files, rows, measure, band)
        band_outputs.append(out)
        band_counts[band] = len(out)

    result = pd.concat(band_outputs, ignore_index=True)
    if len(result) != len(metadata):
        raise AssertionError(f'{csv_path}: output rows {len(result)} != metadata rows {len(metadata)}')
    validate_graph_table(result, csv_path, measure)

    out_dir = output_root / measure / subject
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f'{Path(csv_path).stem}.parquet'
    result.to_parquet(out_path, index=False, compression='zstd')

    return {
        'Subject': subject,
        'Measure': measure,
        'SourceFile': str(Path(csv_path).relative_to(handcrafted_root)),
        'OutputFile': str(out_path.relative_to(output_root)),
        'Rows': len(result),
        'EstimatedEpochs': len(result) // len(BANDS),
        'BandsComplete': int(len(set(result['Band'])) == len(BANDS) and len(set(band_counts.values())) == 1),
        'NaN': int(result[feature_columns(result)].isna().sum().sum()),
        'Inf': int(np.isinf(result[feature_columns(result)].to_numpy(dtype=np.float64)).sum()),
        'DuplicateKeys': 0,
        'Status': 'PASS',
        'Error': '',
    }


def process_subject(subject, handcrafted_root, connectivity_root, output_root):
    csv_files = sorted((handcrafted_root / subject).glob('*.csv'), key=natural_key)
    if not csv_files:
        raise FileNotFoundError(f'No Handcrafted CSV files for {subject}')
    rows = []
    for measure in MEASURES:
        for csv_path in csv_files:
            print(f'[{subject}] {measure}: {csv_path.name}', flush=True)
            rows.append(process_table(
                csv_path, subject, measure, handcrafted_root, connectivity_root, output_root))
    return rows


In [ ]:
# ==========================================================================
# 6. SYNTHETIC UNIT TESTS -- RUN BEFORE REAL DATA
# ==========================================================================
def run_synthetic_tests():
    n = 18

    # Complete graph with unit weights.
    complete = np.ones((1, n, n), dtype=np.float64)
    np.fill_diagonal(complete[0], 0.0)
    f = compute_graph_features(complete)
    degree_cols = [c for c in f if c.endswith('_Degree')]
    strength_cols = [c for c in f if c.endswith('_Strength')]
    cluster_cols = [c for c in f if c.endswith('_Weighted_Clustering')]
    bet_cols = [c for c in f if c.endswith('_Weighted_Betweenness')]
    eig_cols = [c for c in f if c.endswith('_Eigenvector_Centrality')]
    np.testing.assert_allclose(f[degree_cols], 17.0, atol=1e-6)
    np.testing.assert_allclose(f[strength_cols], 17.0, atol=1e-6)
    np.testing.assert_allclose(f[cluster_cols], 1.0, atol=1e-6)
    np.testing.assert_allclose(f[bet_cols], 0.0, atol=1e-6)
    np.testing.assert_allclose(f[eig_cols], 1.0 / np.sqrt(n), atol=1e-6)
    np.testing.assert_allclose(f['Density'], 1.0, atol=1e-6)
    np.testing.assert_allclose(f['Weighted_Global_Efficiency'], 1.0, atol=1e-6)
    np.testing.assert_allclose(f['Characteristic_Path_Length_LCC'], 1.0, atol=1e-6)

    # Empty graph.
    empty = np.zeros((1, n, n), dtype=np.float64)
    f = compute_graph_features(empty)
    np.testing.assert_allclose(f[degree_cols + strength_cols + cluster_cols + bet_cols], 0.0, atol=1e-6)
    np.testing.assert_allclose(f['Density'], 0.0, atol=1e-6)
    np.testing.assert_allclose(f['Weighted_Global_Efficiency'], 0.0, atol=1e-6)
    np.testing.assert_allclose(f['Largest_Component_Fraction'], 1.0 / n, atol=1e-6)
    np.testing.assert_allclose(f['Number_Connected_Components'], n, atol=1e-6)
    np.testing.assert_allclose(f['Path_Length_Defined'], 0.0, atol=1e-6)

    # Unit-weight path graph: analytical normalised betweenness.
    path = np.zeros((1, n, n), dtype=np.float64)
    for i in range(n - 1):
        path[0, i, i + 1] = path[0, i + 1, i] = 1.0
    f = compute_graph_features(path)
    observed = f[bet_cols].to_numpy()[0]
    expected = np.array([2.0 * i * (n - i - 1) / ((n - 1) * (n - 2)) for i in range(n)])
    np.testing.assert_allclose(observed, expected, atol=1e-6)

    # Unit-weight star graph: centre betweenness=1, leaves=0.
    star = np.zeros((1, n, n), dtype=np.float64)
    for i in range(1, n):
        star[0, 0, i] = star[0, i, 0] = 1.0
    f = compute_graph_features(star)
    observed = f[bet_cols].to_numpy()[0]
    np.testing.assert_allclose(observed[0], 1.0, atol=1e-6)
    np.testing.assert_allclose(observed[1:], 0.0, atol=1e-6)

    print('Synthetic graph tests: PASS')


run_synthetic_tests()


In [ ]:
# ==========================================================================
# 7. RUN ONE KAGGLE PATIENT BATCH
# ==========================================================================
input_root, handcrafted_root, connectivity_root = discover_input_roots()
subjects = discover_subjects(handcrafted_root, connectivity_root)

if SUBJECTS_TO_RUN is not None:
    selected = [s.lower() for s in SUBJECTS_TO_RUN]
    missing = sorted(set(selected).difference(subjects))
    if missing:
        raise ValueError(f'Requested subjects not found or incomplete: {missing}')
elif RUN_ALL_SUBJECTS:
    selected = subjects
else:
    lo = BATCH_NUMBER * PATIENTS_PER_BATCH
    selected = subjects[lo:lo + PATIENTS_PER_BATCH]

if not selected:
    raise ValueError(
        f'Batch {BATCH_NUMBER} selects no patient. Available complete subjects: {subjects}')

print('Input root:', input_root)
print('Complete subjects:', subjects)
print('Run mode:', 'ALL SUBJECTS' if RUN_ALL_SUBJECTS and SUBJECTS_TO_RUN is None else 'SELECTED/BATCH')
print(f'Selected subjects ({len(selected)}):', selected)
print('Threshold ratio:', THRESHOLD_RATIO)

if RESET_OUTPUT and OUTPUT_ROOT.exists():
    print('[RESET OUTPUT]', OUTPUT_ROOT)
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

config = {
    'threshold_ratio': THRESHOLD_RATIO,
    'chunk_size': CHUNK_SIZE,
    'measures': list(MEASURES),
    'bands': list(BANDS),
    'channels': CHANNELS_18,
    'selected_subjects': selected,
    'run_all_subjects': RUN_ALL_SUBJECTS,
    'input_root': str(input_root),
}
with open(OUTPUT_ROOT / 'graph_feature_config.json', 'w', encoding='utf-8') as file:
    json.dump(config, file, indent=2)

qc_rows = []
run_rows = []
for subject in selected:
    try:
        subject_rows = process_subject(subject, handcrafted_root, connectivity_root, OUTPUT_ROOT)
        qc_rows.extend(subject_rows)
        run_rows.append({'Subject': subject, 'Status': 'completed', 'Error': ''})
    except Exception as exc:
        traceback.print_exc()
        run_rows.append({'Subject': subject, 'Status': 'failed', 'Error': repr(exc)})

qc = pd.DataFrame(qc_rows)
run_manifest = pd.DataFrame(run_rows)
qc.to_csv(OUTPUT_ROOT / 'graph_feature_qc.csv', index=False)
run_manifest.to_csv(OUTPUT_ROOT / 'graph_feature_run_manifest.csv', index=False)

display(run_manifest)
if not qc.empty:
    display(qc)

if (run_manifest['Status'] != 'completed').any():
    raise RuntimeError('At least one patient failed. Do not save this output as a Kaggle Dataset.')
print('Graph feature extraction batch completed successfully.')


In [ ]:
# ==========================================================================
# 8. FINAL BATCH QC SUMMARY
# ==========================================================================
if qc.empty:
    raise AssertionError('QC table is empty')

summary = (qc.groupby(['Subject', 'Measure'], as_index=False)
             .agg(Files=('SourceFile', 'count'),
                  FeatureRows=('Rows', 'sum'),
                  EstimatedEpochs=('EstimatedEpochs', 'sum'),
                  NaN=('NaN', 'sum'),
                  Inf=('Inf', 'sum'),
                  DuplicateKeys=('DuplicateKeys', 'sum'),
                  CompleteBandFiles=('BandsComplete', 'sum')))
summary['ExpectedCompleteBandFiles'] = summary['Files']
summary['PASS'] = (
    (summary['NaN'] == 0) &
    (summary['Inf'] == 0) &
    (summary['DuplicateKeys'] == 0) &
    (summary['CompleteBandFiles'] == summary['ExpectedCompleteBandFiles'])
)
display(summary)

if not summary['PASS'].all():
    raise AssertionError('Final graph-feature QC failed')

# PLI and wPLI must represent exactly the same epochs for every selected subject.
pivot = summary.pivot(index='Subject', columns='Measure', values='EstimatedEpochs')
if not all(m in pivot.columns for m in MEASURES):
    raise AssertionError('A connectivity measure is missing from the QC summary')
if not (pivot['PLI'] == pivot['wPLI']).all():
    raise AssertionError('PLI and wPLI epoch totals do not match')

print('FINAL GRAPH FEATURE QC: PASS')
print('All selected subjects were completed in this single run.')
print('Save /kaggle/working/GraphFeatures_GAPAWARE_V3 as one new Kaggle Dataset.')
